# Camada Gold - fato_chamados

## O que este notebook faz

Lê as tabelas da Silver, liga tudo ao `chamados` e calcula as colunas que as perguntas precisam (tempos, custo por segundo, jornada, escalonamento, retorno). Depois responde cada pergunta do case com uma query em cima da tabela. Não cria tabela de agregação nem tabela auxiliar.

- Entrada: `rocket_lab.vcredit_silver` (`chamados`, `chamados_hora`, `custos`, `pesquisa`, `clientes`, `motivos`, `canais`, `atendentes`).
- Saída: `rocket_lab.vcredit_gold.fato_chamados`.

## Como o notebook está organizado

| Bloco | Conteúdo |
|---|---|
| 0 | parâmetros (schemas e as duas janelas de tempo) |
| 1 | leitura da Silver, joins e teste de grão |
| 2 | colunas calculadas |
| 3 | jornada, escalonamento e retorno (window functions) |
| 4 | gravação e conferência |
| 5 | perguntas do case, uma query por pergunta |
| 6 | checagens que sustentam as definições abaixo |



## Definições usadas nas análises

Algumas coisas não vêm prontas nos dados e precisaram de uma regra. Estão listadas aqui para ficar claro como cada número foi construído.

Tempos
- Os horários de `chamados_hora` têm prioridade. `tempo_espera` e `tempo_atendimento` do `chamados` só entram quando o horário falta.
- Quando o tempo informado e o calculado pelos horários discordam, a coluna `diverge_espera` ou `diverge_atend` fica verdadeira. Serve de evidência para o relatório de confiabilidade dos dados.

Abandono e nível
- Abandono é chamado sem hora de início e com hora de fim (abandono na fila, regra do grupo).
- Nível do canal: 0 para URA e chatbot, 1 para atendimento inicial, 2 para atendimento especializado.

Custo
- Custo zero, negativo ou ausente fica fora das médias (coluna `custo_valido`).
- Custo por segundo é sempre soma do custo dividida pela soma dos segundos, nunca média de razões.

Atendente
- Só vale em canal humano.

Jornada, escalonamento e retorno
- Cada `id_chamado` aparece uma única vez, então não existe um mesmo chamado passando de nível. O escalonamento é inferido entre chamados do mesmo cliente e do mesmo motivo, abertos em sequência.
- Jornada é essa sequência de chamados. Ela é cortada quando passam mais de 24 horas entre o fim de um chamado e a abertura do próximo (`JANELA_JORNADA_H`).
- Escalonado é o chamado cujo seguinte, na mesma jornada, é de nível maior.
- Retorno é o próximo chamado do cliente, de qualquer motivo, em até 7 dias (`JANELA_RETORNO_D`). Sem retorno é desistência, mesmo motivo é insistência e motivo diferente é problema novo.
- Só entram na análise de retorno os chamados que ainda tinham 7 dias de dados pela frente. Sem isso, os últimos chamados pareceriam desistência só porque não deu tempo de o cliente voltar.

## Bloco 0: definições
As duas janelas são definições de negócio (não dados). Mude e reexecute para testar a sensibilidade das conclusões.

| Parâmetro | Valor | Significado |
|---|---|---|
| `s` | `rocket_lab.vcredit_silver` | schema de origem (somente leitura) |
| `g` | `rocket_lab.vcredit_gold` | schema de destino |
| `JANELA_JORNADA_H` | 24 | até quantas horas entre o fim de um chamado e a abertura do próximo para ainda serem a mesma jornada |
| `JANELA_RETORNO_D` | 7 | até quantos dias para considerar que o cliente "voltou" |


In [0]:
from pyspark.sql import functions as F, Window
s = "rocket_lab.vcredit_silver"
g = "rocket_lab.vcredit_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {g}")
JANELA_JORNADA_H = 24 # até 24h entre o fim de um chamado e a abertura do próximo (mesmo cliente e motivo) = mesma jornada
JANELA_RETORNO_D = 7 # voltou = abriu outro chamado em até 7 dias

## 1. Leitura e joins

Cada tabela da Silver é lida só com as colunas que interessam, e tudo é ligado ao `chamados` com left join. Assim nenhum chamado some se faltar cadastro em outra tabela.

In [0]:
# chamados: tabela base. Descarta só quem não tem id.
ch = (spark.table(f"{s}.chamados").filter(F.col("id_chamado").isNotNull())
      .select("id_chamado", "id_cliente", "motivo", "canal", "resolvido",
              "tempo_espera", "tempo_atendimento", "id_atendente", "flag_atendente_placeholder"))
# chamados_hora: os três horários, com nomes mais curtos
hr = (spark.table(f"{s}.chamados_hora")
      .select("id_chamado",
              F.col("hora_abertura_chamado").alias("abertura"),
              F.col("hora_inicio_atendimento").alias("inicio"),
              F.col("hora_finalizacao_atendimento").alias("fim")))
# custos: soma por chamado, pra garantir 1 linha por id. qtd_custos guarda quantos registros tinha
cu = (spark.table(f"{s}.custos").filter(F.col("id_chamado").isNotNull())
      .groupBy("id_chamado").agg(F.sum("custo").alias("custo"), F.count("*").alias("qtd_custos")))
# pesquisa: uma nota por chamado (se tiver resposta repetida, tira a média)
pe = (spark.table(f"{s}.pesquisa").filter(F.col("id_chamado").isNotNull())
      .groupBy("id_chamado").agg(F.avg("nota_atendimento").alias("nota")))
# clientes: sem nome e e-mail de propósito, não precisa de dado pessoal na Gold
cl = (spark.table(f"{s}.clientes")
      .select("id_cliente", "regiao", "faixa_etaria", "data_cadastro", "dia_vencimento_fatura")
      .withColumn("cliente_identificado", F.lit(True)))

mo = spark.table(f"{s}.motivos").select(F.col("nome_motivo").alias("motivo"), "categoria", "criticidade")
ca = spark.table(f"{s}.canais").select(F.col("nome_canal").alias("canal"), "tipo_canal")

# atendente só vale em canal humano e fora dos placeholders
humano = F.col("canal").isin("atendimento inicial", "atendimento especializado")
ch = ch.withColumn("id_atendente_valido",
        F.when(humano & ~F.coalesce("flag_atendente_placeholder", F.lit(False)), F.col("id_atendente")))

at = (spark.table(f"{s}.atendentes")
      .select(F.col("id_atendente").alias("id_atendente_valido"),
              "nome_atendente", "data_admissao", "data_desligamento"))
# junta tudo a partir dos chamados
f = (ch.join(hr, "id_chamado", "left").join(cu, "id_chamado", "left").join(pe, "id_chamado", "left")
       .join(cl, "id_cliente", "left").join(mo, "motivo", "left").join(ca, "canal", "left")
       .join(at, "id_atendente_valido", "left"))


n_ch = ch.count()
chk = f.agg(F.count("*").alias("n"), F.countDistinct("id_chamado").alias("d")).first()
assert chk["n"] == chk["d"] == n_ch, f"grão quebrado: linhas={chk['n']} ids={chk['d']} chamados={n_ch}"

## 2. Colunas calculadas

Aqui entram datas, nível do canal, abandono, tempos, custo e dados do cliente e do atendente.

Sobre o custo por segundo: média de razões distorce, então a conta certa é soma do custo dividida pela soma dos segundos. Por isso existem `custo_base_seg` e `seg_base_custo`, que guardam custo e segundos só dos chamados que têm os dois valores. Nas queries é só somar as duas e dividir.

In [0]:
ab, ini, fim = F.col("abertura"), F.col("inicio"), F.col("fim")
# tempos calculados pelos horários (em segundos)
espera_hor = F.unix_timestamp("inicio") - F.unix_timestamp("abertura")
atend_hor  = F.unix_timestamp("fim") - F.unix_timestamp("inicio")

f = (f
    # datas
    .withColumn("data_chamado", F.to_date("abertura"))
    .withColumn("mes", F.trunc("data_chamado", "month"))
    # status e nível
    # resolvido em 1/0, pra poder tirar média direto e virar % de resolução
    .withColumn("resolvido_flag", F.when(F.col("resolvido") == "sim", 1).when(F.col("resolvido") == "nao", 0))
    # nível: 0 autosserviço, 1 inicial, 2 especializado
    .withColumn("nivel_canal", F.when(F.col("canal").isin("ura", "chatbot"), 0)
                                .when(F.col("canal") == "atendimento inicial", 1)
                                .when(F.col("canal") == "atendimento especializado", 2))
    # abandono na fila: não começou o atendimento mas tem hora de fim
    .withColumn("abandono", ini.isNull() & fim.isNotNull())
    # tempos: o horário manda, o campo do chamados entra só se faltar horário
    # espera só existe em canal humano; em abandono não tem atendimento
    .withColumn("espera_seg", F.when(humano & ~F.col("abandono"), F.coalesce(espera_hor, F.col("tempo_espera"))))
    .withColumn("atend_seg", F.when(~F.col("abandono"), F.coalesce(atend_hor, F.col("tempo_atendimento"))))
    .withColumn("tempo_total_seg",
        F.when(F.col("espera_seg").isNotNull() | F.col("atend_seg").isNotNull(),
               F.coalesce("espera_seg", F.lit(0)) + F.coalesce("atend_seg", F.lit(0))))
    # marca onde o tempo informado não bate com o dos horários
    .withColumn("diverge_espera", F.coalesce(espera_hor != F.col("tempo_espera"), F.lit(False)))
    .withColumn("diverge_atend", F.coalesce(atend_hor != F.col("tempo_atendimento"), F.lit(False)))

    # custo: só conta se for maior que zero
    .withColumn("custo_valido", F.when(F.col("custo") > 0, F.col("custo")))
    .withColumn("custo_por_seg", F.when(F.col("atend_seg") > 0, F.col("custo_valido") / F.col("atend_seg")))
    # base do custo por segundo agregado (soma / soma)
    .withColumn("custo_base_seg", F.when(F.col("custo_valido").isNotNull() & (F.col("atend_seg") > 0), F.col("custo_valido")))
    .withColumn("seg_base_custo", F.when(F.col("custo_valido").isNotNull() & (F.col("atend_seg") > 0), F.col("atend_seg")))
    # cliente
    .withColumn("cliente_identificado", F.coalesce("cliente_identificado", F.lit(False)))
    .withColumn("ano_cadastro", F.year("data_cadastro"))
    .withColumn("meses_desde_cadastro", F.floor(F.months_between("data_chamado", "data_cadastro")))
    
    # atendente atendendo antes de ser admitido ou depois de desligado
    .withColumn("atendente_fora_do_periodo", F.coalesce(
        (F.col("data_chamado") < F.col("data_admissao")) | (F.col("data_chamado") > F.col("data_desligamento")),
        F.lit(False)))
)

## 3. Jornada, escalonamento e retorno

1. Pega todos os chamados do mesmo cliente e motivo, ordenados por abertura. Quando o próximo abre mais de 24h depois do fim do anterior, começa uma jornada nova. A soma acumulada dessas viradas numera as jornadas, e o hash dá o `id_jornada`.
2. Dentro da jornada: a ordem do chamado, quantos chamados ela tem (mais de um = recorrente) e o nível do chamado seguinte. Se o seguinte é de nível maior, esse chamado foi escalonado.
3. Somas por jornada (custo e tempos) repetidas em todas as linhas dela. Nas queries usa-se `ordem_na_jornada = 1` pra contar cada jornada uma vez.
4. Retorno: olha o próximo chamado do cliente, de qualquer motivo. Sem próximo em 7 dias é desistência, mesmo motivo é insistência, outro motivo é problema novo. Escalonado é continuação da jornada, então fica separado.

In [0]:
lim = f.agg(F.min("abertura"), F.max("abertura")).first()
MAX_ABERTURA = lim[1]
print("dados de", lim[0], "até", MAX_ABERTURA)
 

# chamado com dados faltando vira uma partição só dele (id próprio), pra não se juntar com ninguém
valido_cli = F.col("id_cliente").isNotNull() & F.col("abertura").isNotNull()
valido_cm  = valido_cli & F.col("motivo").isNotNull()
sozinho = F.concat(F.lit("sem_dados|"), F.col("id_chamado").cast("string"))
k_cm  = F.when(valido_cm, F.concat_ws("|", "id_cliente", "motivo")).otherwise(sozinho)
k_cli = F.when(valido_cli, F.col("id_cliente")).otherwise(sozinho)
 
w_cm  = Window.partitionBy(k_cm).orderBy("abertura", "id_chamado")    # mesmo cliente + motivo
w_cli = Window.partitionBy(k_cli).orderBy("abertura", "id_chamado")   # mesmo cliente (qualquer motivo)
 
f = (f
    .withColumn("contato_valido", valido_cm)
    .withColumn("fim_ref", F.coalesce("fim", "inicio", "abertura"))
    # (1) jornada: corta quando passa JANELA_JORNADA_H entre o fim do anterior e a abertura atual
    .withColumn("fim_anterior", F.lag("fim_ref").over(w_cm))
    .withColumn("horas_desde_anterior", (F.unix_timestamp("abertura") - F.unix_timestamp("fim_anterior")) / 3600)
    .withColumn("nova_jornada", F.when(F.col("fim_anterior").isNull() | (F.col("horas_desde_anterior") > JANELA_JORNADA_H), 1).otherwise(0))
    .withColumn("n_jornada", F.sum("nova_jornada").over(w_cm.rowsBetween(Window.unboundedPreceding, 0)))
    .withColumn("id_jornada", F.xxhash64(k_cm, F.col("n_jornada")))
)
 
w_j  = Window.partitionBy("id_jornada").orderBy("abertura", "id_chamado")
w_jt = Window.partitionBy("id_jornada")
 
f = (f
    # (2) dentro da jornada
    .withColumn("ordem_na_jornada", F.row_number().over(w_j))
    .withColumn("qtd_chamados_jornada", F.count("*").over(w_jt))
    .withColumn("flag_recorrente", F.col("qtd_chamados_jornada") > 1)
    .withColumn("nivel_proximo", F.lead("nivel_canal").over(w_j))
    .withColumn("flag_escalonado", F.coalesce(F.col("nivel_proximo") > F.col("nivel_canal"), F.lit(False)))
    # soma das iterações de cada jornada (repetida em todas as linhas da jornada)
    .withColumn("custo_jornada", F.sum("custo_valido").over(w_jt))
    .withColumn("espera_jornada_seg", F.sum("espera_seg").over(w_jt))
    .withColumn("atend_jornada_seg", F.sum("atend_seg").over(w_jt))
    .withColumn("tempo_jornada_seg", F.sum("tempo_total_seg").over(w_jt))
    # (3) retorno do cliente (qualquer motivo)
    .withColumn("proximo_motivo", F.lead("motivo").over(w_cli))
    .withColumn("proximo_abertura", F.lead("abertura").over(w_cli))
    .withColumn("dias_ate_proximo", (F.unix_timestamp("proximo_abertura") - F.unix_timestamp("fim_ref")) / 86400)
    .withColumn("retorno_observavel",
        F.expr(f"fim_ref <= timestamp'{MAX_ABERTURA}' - INTERVAL {JANELA_RETORNO_D} DAYS"))
    .withColumn("tipo_retorno",
        F.when(~F.col("contato_valido"), "indeterminado")
         .when(F.col("flag_escalonado"), "escalonado")      # continuação da jornada, não é insistência
         .when(F.col("proximo_abertura").isNull() | (F.col("dias_ate_proximo") > JANELA_RETORNO_D), "sem_retorno")
         .when(F.col("proximo_motivo") == F.col("motivo"), "mesmo_motivo")
         .otherwise("novo_motivo"))
)

## 4. Gravação
Grava a tabela em Delta com overwrite.
A ordem das colunas é definida na lista `ordem`, agrupada por assunto (chaves, chamado, tempos, custo, cliente, atendente, jornada, retorno). As colunas auxiliares que só serviram de passo intermediário são descartadas.

Em seguida imprime umas contagens pra conferir se nada saiu estranho.


In [0]:
# ordem das colunas na tabela final, agrupadas por assunto
ordem = [
    # chaves
    "id_chamado", "id_cliente", "id_jornada",
    # o chamado
    "motivo", "categoria", "criticidade", "canal", "tipo_canal", "nivel_canal",
    "resolvido", "resolvido_flag",
    # datas e horários
    "abertura", "inicio", "fim", "data_chamado", "mes",
    # tempos
    "tempo_espera", "tempo_atendimento",           # como vieram do chamados
    "espera_seg", "atend_seg", "tempo_total_seg",  # os usados nas análises
    "diverge_espera", "diverge_atend", "abandono",
    # custo e nota
    "custo", "qtd_custos", "custo_valido", "custo_por_seg",
    "custo_base_seg", "seg_base_custo", "nota",
    # cliente
    "cliente_identificado", "regiao", "faixa_etaria", "data_cadastro",
    "ano_cadastro", "meses_desde_cadastro", "dia_vencimento_fatura",
    # atendente
    "id_atendente", "flag_atendente_placeholder", "id_atendente_valido",
    "nome_atendente", "data_admissao", "data_desligamento", "atendente_fora_do_periodo",
    # jornada e escalonamento
    "contato_valido", "ordem_na_jornada", "qtd_chamados_jornada", "flag_recorrente",
    "horas_desde_anterior", "nivel_proximo", "flag_escalonado",
    "custo_jornada", "espera_jornada_seg", "atend_jornada_seg", "tempo_jornada_seg",
    # retorno
    "proximo_motivo", "dias_ate_proximo", "retorno_observavel", "tipo_retorno",
]
# colunas auxiliares que só serviram de passo intermediário (não vão pra tabela)
auxiliares = ["fim_anterior", "nova_jornada", "n_jornada", "fim_ref", "proximo_abertura"]
# se sobrar alguma coluna fora da lista, ela vai pro final em vez de sumir
resto = [c for c in f.columns if c not in ordem and c not in auxiliares]

(f.select(*[c for c in ordem if c in f.columns], *resto)
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{g}.fato_chamados"))

t = spark.table(f"{g}.fato_chamados")
print("chamados:", t.count(),
      "| sem horário:", t.filter("abertura is null").count(),
      "| cliente não identificado:", t.filter("not cliente_identificado").count(),
      "| sem custo:", t.filter("custo is null").count(),
      "| custo <= 0:", t.filter("custo <= 0").count(),
      "| sem nota:", t.filter("nota is null").count(),
      "| abandonos:", t.filter("abandono").count(),
      "| jornadas recorrentes:", t.filter("flag_recorrente and ordem_na_jornada = 1").count(),
      "| escalonados:", t.filter("flag_escalonado").count())
display(t.limit(20))

## 5. Perguntas do case

Daqui pra baixo nada é gravado. A view temporária `fato` só dá um nome curto pra usar nos SQLs.

In [0]:
spark.table(f"{g}.fato_chamados").createOrReplaceTempView("fato")

### Custo e tempo médios (geral e por segundo): por tipo, nível e resolvido

In [0]:
# por tipo
display(spark.sql("""
    SELECT categoria, motivo,
           COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(try_divide(SUM(custo_base_seg), SUM(seg_base_custo)), 6) AS custo_por_seg,
           ROUND(AVG(espera_seg), 1) AS espera_media_seg,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato
    GROUP BY categoria, motivo
    ORDER BY categoria, motivo"""))
 
# por nível (0 = autosserviço)
display(spark.sql("""
    SELECT nivel_canal,
           COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(try_divide(SUM(custo_base_seg), SUM(seg_base_custo)), 6) AS custo_por_seg,
           (ROUND(AVG(espera_seg), 1)) AS espera_media_seg,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato
    GROUP BY nivel_canal
    ORDER BY nivel_canal"""))
 
 
# por resolvido / não resolvido
display(spark.sql("""
    SELECT resolvido,
           COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(try_divide(SUM(custo_base_seg), SUM(seg_base_custo)), 6) AS custo_por_seg,
           ROUND(AVG(espera_seg), 1) AS espera_media_seg,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato
    GROUP BY resolvido
    ORDER BY resolvido"""))

### Recorrência: custos, cruzamento com tipo e nível

In [0]:
display(spark.sql("""
    SELECT flag_recorrente, COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(try_divide(SUM(custo_base_seg), SUM(seg_base_custo)), 6) AS custo_por_seg,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato WHERE contato_valido
    GROUP BY flag_recorrente"""))
 
display(spark.sql("""
    SELECT categoria, flag_recorrente, COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato WHERE contato_valido
    GROUP BY categoria, flag_recorrente
    ORDER BY categoria, flag_recorrente"""))
 
display(spark.sql("""
    SELECT nivel_canal, flag_recorrente, COUNT(*) AS chamados,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg
    FROM fato WHERE contato_valido
    GROUP BY nivel_canal, flag_recorrente
    ORDER BY nivel_canal, flag_recorrente"""))

### Média da soma dos tempos de cada jornada
O filtro `ordem_na_jornada = 1` pega uma linha por jornada (as somas se repetem em todas as linhas da jornada).

In [0]:
display(spark.sql("""
    SELECT flag_recorrente, COUNT(*) AS jornadas,
           ROUND(AVG(qtd_chamados_jornada), 2) AS chamados_por_jornada,
           ROUND(AVG(custo_jornada), 4) AS custo_medio_jornada,
           ROUND(AVG(espera_jornada_seg), 1) AS espera_media_jornada_seg,
           ROUND(AVG(atend_jornada_seg), 1) AS atend_medio_jornada_seg,
           ROUND(AVG(tempo_jornada_seg), 1) AS tempo_medio_jornada_seg
    FROM fato WHERE contato_valido AND ordem_na_jornada = 1
    GROUP BY flag_recorrente"""))
 
display(spark.sql("""
    SELECT categoria, COUNT(*) AS jornadas,
           ROUND(100 * AVG(CAST(flag_recorrente AS INT)), 2) AS pct_recorrentes,
           ROUND(AVG(custo_jornada), 4) AS custo_medio_jornada,
           ROUND(AVG(tempo_jornada_seg), 1) AS tempo_medio_jornada_seg
    FROM fato WHERE contato_valido AND ordem_na_jornada = 1
    GROUP BY categoria ORDER BY categoria"""))
 
display(spark.sql("""
    SELECT nivel_canal AS nivel_de_entrada, COUNT(*) AS jornadas,
           ROUND(100 * AVG(CAST(flag_recorrente AS INT)), 2) AS pct_recorrentes,
           ROUND(AVG(custo_jornada), 4) AS custo_medio_jornada,
           ROUND(AVG(tempo_jornada_seg), 1) AS tempo_medio_jornada_seg
    FROM fato WHERE contato_valido AND ordem_na_jornada = 1
    GROUP BY nivel_canal ORDER BY nivel_canal"""))

### Nota baixa leva a desistência ou insistência?
`sem_retorno` = desistência | `mesmo_motivo` = insistência | `novo_motivo` = problema novo | `escalonado` = continuação da jornada.

In [0]:
display(spark.sql("""
    SELECT nota_int, tipo_retorno, COUNT(*) AS n,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY nota_int), 1) AS pct
    FROM (SELECT CAST(ROUND(nota) AS INT) AS nota_int, tipo_retorno
          FROM fato
          WHERE nota IS NOT NULL AND contato_valido AND retorno_observavel)
    GROUP BY nota_int, tipo_retorno
    ORDER BY nota_int, tipo_retorno"""))

### Clientes antigos e novos: nota e % resolvidos por ano de cadastro

In [0]:
display(spark.sql("""
    SELECT ano_cadastro,
           COUNT(*) AS chamados,
           COUNT(nota) AS respostas,
           ROUND(AVG(nota), 3) AS nota_media,
           ROUND(100 * AVG(resolvido_flag), 2) AS pct_resolvido
    FROM fato WHERE cliente_identificado
    GROUP BY ano_cadastro ORDER BY ano_cadastro"""))

### Chamados por dia de vencimento da fatura

In [0]:
display(spark.sql("""
    SELECT dia_vencimento_fatura, COUNT(*) AS chamados
    FROM fato WHERE cliente_identificado
    GROUP BY dia_vencimento_fatura ORDER BY dia_vencimento_fatura"""))

### Abandono por perfil
A primeira consulta mostra em quais canais o abandono aparece. As demais olham só canais humanos, onde existe fila.

In [0]:
display(spark.sql("""
    SELECT tipo_canal, COUNT(*) AS chamados,
           SUM(CAST(abandono AS INT)) AS abandonos,
           ROUND(100 * AVG(CAST(abandono AS INT)), 2) AS pct_abandono
    FROM fato GROUP BY tipo_canal"""))
 
display(spark.sql("""
    SELECT faixa_etaria, COUNT(*) AS chamados, SUM(CAST(abandono AS INT)) AS abandonos,
           ROUND(100 * AVG(CAST(abandono AS INT)), 2) AS pct_abandono
    FROM fato WHERE tipo_canal = 'humano'
    GROUP BY faixa_etaria ORDER BY faixa_etaria"""))
 
display(spark.sql("""
    SELECT regiao, COUNT(*) AS chamados, SUM(CAST(abandono AS INT)) AS abandonos,
           ROUND(100 * AVG(CAST(abandono AS INT)), 2) AS pct_abandono
    FROM fato WHERE tipo_canal = 'humano'
    GROUP BY regiao ORDER BY regiao"""))
 
display(spark.sql("""
    SELECT categoria, COUNT(*) AS chamados, SUM(CAST(abandono AS INT)) AS abandonos,
           ROUND(100 * AVG(CAST(abandono AS INT)), 2) AS pct_abandono
    FROM fato WHERE tipo_canal = 'humano'
    GROUP BY categoria ORDER BY categoria"""))

### Taxa de escalonamento (e por tipo)
 Nível 2 fica de fora (não tem para onde subir). A matriz mostra para qual nível foi o próximo chamado da jornada (destino nulo = a jornada terminou ali).

In [0]:
display(spark.sql("""
    SELECT nivel_canal, COUNT(*) AS chamados,
           SUM(CAST(flag_escalonado AS INT)) AS escalonados,
           ROUND(100 * AVG(CAST(flag_escalonado AS INT)), 2) AS pct_escalonado
    FROM fato WHERE contato_valido AND nivel_canal < 2
    GROUP BY nivel_canal ORDER BY nivel_canal"""))
 
display(spark.sql("""
    SELECT categoria, motivo, COUNT(*) AS chamados,
           SUM(CAST(flag_escalonado AS INT)) AS escalonados,
           ROUND(100 * AVG(CAST(flag_escalonado AS INT)), 2) AS pct_escalonado
    FROM fato WHERE contato_valido AND nivel_canal < 2
    GROUP BY categoria, motivo ORDER BY pct_escalonado DESC"""))
 
display(spark.sql("""
    SELECT nivel_canal AS nivel_origem, nivel_proximo AS nivel_destino, COUNT(*) AS n
    FROM fato WHERE contato_valido
    GROUP BY nivel_canal, nivel_proximo ORDER BY nivel_canal, nivel_proximo"""))

### Taxa de resolução URA x chatbot

In [0]:
display(spark.sql("""
    SELECT canal, COUNT(*) AS chamados,
           ROUND(100 * AVG(resolvido_flag), 2) AS pct_resolvido
    FROM fato WHERE canal IN ('ura', 'chatbot')
    GROUP BY canal"""))

### Custo por mês, causa da variação e evoluções
A primeira consulta traz custo médio, custo absoluto e volume por mês. Os cruzamentos por tipo e por nível mostram de onde vem a variação.

In [0]:
display(spark.sql("""
    SELECT mes, COUNT(*) AS chamados,
           ROUND(SUM(custo_valido), 2) AS custo_total,
           ROUND(AVG(custo_valido), 4) AS custo_medio,
           ROUND(try_divide(SUM(custo_base_seg), SUM(seg_base_custo)), 6) AS custo_por_seg
    FROM fato WHERE mes IS NOT NULL
    GROUP BY mes ORDER BY mes"""))
 
display(spark.sql("""
    SELECT mes, categoria, COUNT(*) AS chamados,
           ROUND(SUM(custo_valido), 2) AS custo_total,
           ROUND(AVG(custo_valido), 4) AS custo_medio
    FROM fato WHERE mes IS NOT NULL
    GROUP BY mes, categoria ORDER BY mes, categoria"""))
 
display(spark.sql("""
    SELECT mes, nivel_canal, COUNT(*) AS chamados,
           ROUND(SUM(custo_valido), 2) AS custo_total,
           ROUND(AVG(custo_valido), 4) AS custo_medio
    FROM fato WHERE mes IS NOT NULL
    GROUP BY mes, nivel_canal ORDER BY mes, nivel_canal"""))

### Atendentes ativos por mês x custo e volume
Ativo no mês = admissão até o fim do mês e desligamento nulo ou depois do início do mês.

In [0]:
display(spark.sql(f"""
    WITH lim AS (SELECT MIN(mes) AS a, MAX(mes) AS b FROM fato),
    meses AS (SELECT explode(sequence(a, b, interval 1 month)) AS mes FROM lim),
    ativos AS (
        SELECT m.mes, COUNT(DISTINCT a.id_atendente) AS atendentes_ativos
        FROM meses m
        LEFT JOIN {s}.atendentes a
          ON a.data_admissao <= last_day(m.mes)
         AND (a.data_desligamento IS NULL OR a.data_desligamento >= m.mes)
        GROUP BY m.mes),
    mensal AS (
        SELECT mes, COUNT(*) AS chamados,
               ROUND(SUM(custo_valido), 2) AS custo_total,
               ROUND(AVG(custo_valido), 4) AS custo_medio,
               COUNT(DISTINCT id_atendente_valido) AS atendentes_com_chamado
        FROM fato WHERE mes IS NOT NULL GROUP BY mes)
    SELECT mes, atendentes_ativos, atendentes_com_chamado, chamados,
           try_divide(chamados, atendentes_ativos) AS chamados_por_ativo,
           custo_total, custo_medio
    FROM ativos JOIN mensal USING (mes)
    ORDER BY mes"""))

### Por atendente: nota, tempo e número de atendimentos

In [0]:
display(spark.sql("""
    SELECT id_atendente_valido, nome_atendente,
           COUNT(*) AS atendimentos,
           ROUND(AVG(nota), 3) AS nota_media,
           ROUND(AVG(atend_seg), 1) AS atend_medio_seg,
           ROUND(AVG(espera_seg), 1) AS espera_media_seg,
           SUM(CAST(atendente_fora_do_periodo AS INT)) AS fora_do_periodo
    FROM fato WHERE id_atendente_valido IS NOT NULL
    GROUP BY id_atendente_valido, nome_atendente
    ORDER BY atendimentos DESC"""))

### Resumo de inconsistências

O que cada linha mede:

- Sem hora de abertura: o chamado não tem horário de abertura. Fica fora de qualquer análise por data.
- Sem hora de início: não houve início de atendimento.
- Abandono na fila: sem hora de início e com hora de fim.
- Cliente não identificado: o `id_cliente` não existe em `clientes`, porque a Silver descartou idades fora de 18 a 110.
- Sem motivo/categoria: o motivo não foi encontrado em `motivos`.
- Sem custo: o chamado não tem registro de custo.
- Custo <= 0: custo zero ou negativo, que fica fora das médias (`custo_valido`).
- Mais de um registro de custo: chamado com mais de um custo.
- Sem nota de pesquisa: o cliente não respondeu a pesquisa.
- Tempo de espera ou de atendimento diverge dos horários: o tempo informado no `chamados` não bate com o calculado por `chamados_hora`.
- Canal humano sem atendente válido: atendimento humano sem `id_atendente` ou com id placeholder (0, 99, 999).
- Atendente fora do período de vínculo: o atendimento aconteceu antes da admissão ou depois do desligamento.

In [0]:
# Resumo de inconsistências: quantos chamados caem em cada problema e o % sobre o total
display(spark.sql("""
    SELECT t.inconsistencia, t.qtd, ROUND(100 * t.qtd / a.total, 2) AS pct
    FROM (
        SELECT COUNT(*) AS total,
               SUM(CAST(abertura IS NULL AS INT))                         AS c1,
               SUM(CAST(inicio IS NULL AS INT))                           AS c2,
               SUM(CAST(abandono AS INT))                                 AS c3,
               SUM(CAST(NOT cliente_identificado AS INT))                 AS c4,
               SUM(CAST(categoria IS NULL AS INT))                        AS c5,
               SUM(CAST(custo IS NULL AS INT))                            AS c6,
               COALESCE(SUM(CAST(custo <= 0 AS INT)), 0)                  AS c7,
               COALESCE(SUM(CAST(qtd_custos > 1 AS INT)), 0)              AS c8,
               SUM(CAST(nota IS NULL AS INT))                             AS c9,
               SUM(CAST(diverge_espera AS INT))                           AS c10,
               SUM(CAST(diverge_atend AS INT))                            AS c11,
               SUM(CAST(nivel_canal > 0 AND id_atendente_valido IS NULL AS INT)) AS c12,
               SUM(CAST(atendente_fora_do_periodo AS INT))                AS c13,
               SUM(CAST(NOT contato_valido AS INT))                       AS c14
        FROM fato) a
    LATERAL VIEW stack(14,
        'sem hora de abertura',                      c1,
        'sem hora de início',                        c2,
        'abandono na fila',                          c3,
        'cliente não identificado',                  c4,
        'sem motivo/categoria',                      c5,
        'sem custo',                                 c6,
        'custo <= 0',                                c7,
        'mais de um registro de custo',              c8,
        'sem nota de pesquisa',                      c9,
        'tempo de espera diverge dos horários',      c10,
        'tempo de atendimento diverge dos horários', c11,
        'canal humano sem atendente válido',         c12,
        'atendente fora do período de vínculo',      c13,
        'fora da análise de jornada (dados incompletos)', c14
    ) t AS inconsistencia, qtd
    ORDER BY t.qtd DESC"""))